# Step 6: Evaluation and diagnostics

**The only place final numbers are produced.** Steps 3a (baselines), 3b (Group 10 NRMS), 4 (deep hybrid, the main model) and 5 (its ablations) write prediction files to Drive; this notebook reads them, checks them, and computes every table, figure and number in the report with one evaluator (`shared/evaluator.py`, verified against a brute-force implementation in `tests/test_evaluator.py`). Nothing is trained here.

**To run:** Runtime → Run all (a CPU runtime is enough). It reads `MyDrive/BT4222_G11/features/v0.1/` and `MyDrive/BT4222_G11/predictions/real/`, and writes everything to `MyDrive/BT4222_G11/step6_outputs/real/` (cleared at the start of each run). The filled report is `step6_outputs/real/report_sections_filled.md`.

Rules (`docs/evaluation_spec.md`):

* Validation drove every choice; test (14 Nov) is read once for confirmation. Error analysis and the balance rule use validation.
* Cold at threshold *t* means fewer than *t* prior exposures at impression time. *t* = 10 is primary; 5 and 20 are sensitivity checks.
* Every interval is a 95% bootstrap over **users** (all impressions of a resampled user move together), paired across models on the same impressions.
* Every table carries its support (eligible impressions and clicks).

| Section | Report use |
|---|---|
| 1. Data support audit | Layer 0: is there enough cold data; threshold justification |
| 2. Prediction file checks | Reproducibility |
| 3. Headline table | Layers 1–2: baselines and NRMS vs the main model, train / val / test |
| 4. Threshold robustness | Cold vs warm at 5 / 10 / 20 |
| 5. Metric robustness, exposure | Layers 3–4: popularity-neutral metrics; who gets the first screen |
| 6. Balance | Layer 5: trade-off curve, slot-quota sweep, balance rule |
| 7. Ablation | Which component of the deep model is responsible |
| 8. Segments | For whom and when |
| 9. Stress slates | Secondary, resampled |
| 10. Error analysis | Why the model still fails |
| 11. Business numbers | Business insights |
| 12. Research question | Direct answer, parts 1 and 2 |
| 13. Export | Tables, figures, `report_numbers.json`, filled report sections |


## 0. Setup

In [ ]:
import os, sys, subprocess, glob, json, time, warnings

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from shared import schema as S, feature_groups as FG
from shared import evaluator as E, segments as SG, audit as A, diagnostics as D, report_plots as P
from shared.predictions import read_predictions
from shared.report_io import Exporter

DATA_KIND = os.environ.get("G11_DATA_KIND", "REAL")       # SYNTHETIC only for local development
if DATA_KIND == "REAL":
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
else:
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"synthetic_v{S.SCHEMA_VERSION}", "learnable")
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
OUT_DIR = os.path.join(DATA_ROOT, "step6_outputs", DATA_KIND.lower())
version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"

# ---- Frozen settings (docs/evaluation_spec.md) ----
READ_TEST = True                                          # every choice was frozen on validation; test is read once here
PRIMARY_T = S.PRIMARY_COLD                                 # 10
THRESHOLDS = S.COLD_THRESHOLDS                             # (5, 10, 20)
N_BOOT = int(os.environ.get("G11_N_BOOT", 1000))
BOOT_SEED = 0
QUOTAS = (0.0, 0.1, 0.2, 0.3)
MAX_WARM_DROP = (0.01, 0.02, 0.05)                         # tolerances x reported for the balance rule
STRESS = dict(slate_size=10, warm_share=0.4, seeds=(0, 1, 2))
ERROR_N = 30

SPLITS = ["train", "val"] + (["test"] if READ_TEST else [])
EVAL_SPLITS = [s for s in SPLITS if s != "train"]          # splits with intervals
REPORT_SPLIT = "test" if READ_TEST else "val"              # split shown in figures
# Ablation variants of the main deep model (step 5). CA = candidate-aware attention, IDM = ID masking,
# H = content smoothed CTR added as a feature. Every variant keeps the attention news and user encoders.
FG.ABLATIONS = {
    "full":                   ["T1", "T2", "T3", "U", "CA", "IDM"],
    "no_candidate_attention": ["T1", "T2", "T3", "U", "IDM"],
    "no_id_masking":          ["T1", "T2", "T3", "U", "CA"],
    "no_t1":                  ["T2", "T3", "U", "CA", "IDM"],
    "no_t2":                  ["T1", "T3", "U", "CA", "IDM"],
    "no_t3":                  ["T1", "T2", "U", "CA", "IDM"],
    "no_u":                   ["T1", "T2", "T3", "CA", "IDM"],
    "full_plus_smoothing":    ["T1", "T2", "T3", "U", "CA", "IDM", "H"],
}

import shutil
shutil.rmtree(OUT_DIR, ignore_errors=True)                 # outputs are regenerated in full on every run
P.setup_style()
X = Exporter(OUT_DIR)
print(f"DATA_KIND={DATA_KIND} | READ_TEST={READ_TEST} | splits={SPLITS} | n_boot={N_BOOT} | out={OUT_DIR}")

## 1. Data support audit (step 0: before any model is read)

Counts clicked cold candidates and cold-click impressions per split and threshold, and recomputes prior exposures from the row log to check the cold flags. The threshold justification uses **prior exposures at impression time** (what the cold flag uses), not total impressions over the window.

In [ ]:
META_COLS = ["row_id", "impression_id", "user_id", "news_id", "time", "split", "label", "slate_size", "prior_exposures",
             "is_zero_exposure", "is_cold_5", "is_cold_10", "is_cold_20", "history_length", "category", "subcategory"]
meta = {}
for split in S.SPLITS:      # test meta has no model scores in it, so loading it for support counts is safe
    m = pd.read_parquet(S.meta_path(FEATURE_ROOT, split))
    meta[split] = m[[c for c in META_COLS if c in m.columns]].reset_index(drop=True)
meta_all = pd.concat([meta[s] for s in S.SPLITS], ignore_index=True)

structure = pd.DataFrame({s: A.check_meta(meta[s], s) for s in S.SPLITS}).T
X.table(structure, "audit_meta_structure", "Structural checks on the metadata tables")
display(structure)

flags = A.check_cold_flags(meta_all)
X.table(flags, "audit_cold_flags", "Stored prior_exposures and cold flags vs an independent recomputation")
display(flags.round(4))
if (flags["prior_exposures_agree"] < 0.999).any():
    warnings.warn("prior_exposures differ from the recomputation (strictly earlier timestamps, counted from the "
                  "start of the log). Check the counting rule before reading results.")

In [ ]:
support = A.cold_support(meta)
X.table(support, "support_cold_clicks", "Cold-click support per split and threshold (cold = prior exposures < t)")
for (split, t), row in support.iterrows():
    X.number(f"support.{split}.t{t}.cold_click_impressions", int(row["cold_click_impressions"]), "{:,}")
    X.number(f"support.{split}.t{t}.cold_clicks", int(row["cold_clicks"]), "{:,}")
    X.number(f"support.{split}.t{t}.cold_click_share", row["cold_click_share"], "{:.1%}")
    X.number(f"support.{split}.t{t}.warm_click_share", 1 - row["cold_click_share"], "{:.1%}")
    X.number(f"support.{split}.t{t}.cold_row_share", row["cold_row_share"], "{:.1%}")
    X.number(f"support.{split}.t{t}.ci_halfwidth", row["ci_halfwidth_approx"], "{:.3f}")
display(support.round(4))
thin = support[support["cold_click_impressions"] < 300]
if len(thin):
    print("Fewer than 300 cold-click impressions (interval half-width above ~0.045):")
    display(thin[["cold_click_impressions", "ci_halfwidth_approx"]])

In [ ]:
volume = A.threshold_volume(meta_all)
noise = A.ctr_noise_curve(meta_all)
X.table(volume, "threshold_volume", "Time and traffic needed to leave the cold state (prior exposures)")
X.table(noise, "threshold_noise_curve", "Error of early CTR as a predictor of later CTR, by prior exposures")
X.number("volume.median_exposures_per_article_day", volume.attrs["median_exposures_per_article_day"], "{:.0f}")
for t, row in volume.iterrows():
    X.number(f"volume.t{t}.median_hours", row["median_hours_to_t"], "{:.1f}")
    X.number(f"volume.t{t}.share_reaching", row["share_reaching_t"], "{:.1%}")
    X.number(f"volume.t{t}.share_within_24h", row["share_reaching_t_within_24h"], "{:.1%}")
for n in (5, 10, 20):
    if n in noise.index:
        X.number(f"noise.n{n}.mae", noise.loc[n, "mae"], "{:.4f}")
        X.number(f"noise.n{n}.rel", noise.loc[n, "mae_relative_to_n1"], "{:.2f}")
display(volume.round(3), noise.round(4))
fig = P.threshold_figure(noise, volume)
P.save(fig, X.figure_stem("fig_threshold_justification", "Threshold justification: early-CTR noise and time to warm"))
plt.show()

## 2. Prediction file inventory and checks

Only frozen choices are evaluated: files whose metadata say `selected_on = val`. Tuning-grid files are ignored. Every file is checked for row_id alignment, duplicates and finite scores before scoring; scores are joined to meta on `row_id`, never by position.

In [ ]:
KIND = {"random": "baseline", "popularity": "baseline", "category_ctr": "baseline",
        "candidate_smoothed_ctr": "baseline", "content_smoothed_ctr": "baseline",
        "nrms_g10": "reference", "deep_hybrid": "main"}
REFERENCES = ["category_ctr", "candidate_smoothed_ctr", "nrms_g10"]   # every headline delta is against these
MAIN = "deep_hybrid"

def wanted(model_id):
    """Only the baselines, Group 10's NRMS, the main model and its ablation variants are evaluated."""
    return model_id in KIND or model_id.startswith("deep_")

inventory, files = [], {}
for path in sorted(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__*.parquet"))):
    pred, info = read_predictions(path)
    if info.get("selected_on") != "val" or info.get("split") not in S.SPLITS or not wanted(info.get("model_id", "")):
        continue
    model_id, split = info["model_id"], info["split"]
    seed = info.get("seed")
    key = model_id if seed is None else f"{model_id}@seed{seed}"
    chk = A.check_predictions(meta[split], pred)
    inventory.append({"model": key, "split": split, "config_id": info["config_id"], "variant": info.get("variant"),
                      "schema": info.get("schema_version"), "created": info.get("created_utc"), **chk})
    if chk["ok"] and split in SPLITS:
        files[(key, split)] = path
inventory = pd.DataFrame(inventory).set_index(["model", "split"]).sort_index()
X.table(inventory.drop(columns=["created"]), "audit_prediction_files", "Prediction files read by step 6")
display(inventory)
bad = inventory[~inventory["ok"]]
assert bad.empty, f"prediction files failed checks:\n{bad}"
MODELS = sorted({m for m, _ in files})
def ablation_model(variant):
    return MAIN if variant == "full" else f"deep_{variant}"
ABLATION_MODELS = {ablation_model(v): v for v in FG.ABLATIONS if ablation_model(v) in MODELS}
print("models:", MODELS)
print("ablation variants:", list(ABLATION_MODELS.values()))
assert MAIN in MODELS, f"no {MAIN} prediction files in {os.path.join(PRED_ROOT, DATA_KIND.lower())}"

In [ ]:
_score_cache = {}
def scores(model, split):
    if (model, split) not in _score_cache:
        pred, _ = read_predictions(files[(model, split)])
        _score_cache[(model, split)] = A.align_scores(meta[split], pred)
    return _score_cache[(model, split)]

_pi_cache = {}
def per_imp(model, split, t=PRIMARY_T):
    # per-impression table; cached for the primary threshold on val/test only (memory)
    key = (model, split, t)
    if key in _pi_cache:
        return _pi_cache[key]
    pi = E.per_impression_metrics(meta[split], scores(model, split), cold_threshold=t)
    if t == PRIMARY_T and split != "train":
        _pi_cache[key] = pi
    return pi

def has(model, split):
    return (model, split) in files

HEAD = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10", "zero_recall@10",
        "cold_auc_within", "cold_auc_vs_all", "cold_ndcg@10", "popdebiased_ndcg@10_g0.5", "popdebiased_ndcg@10_g1",
        "cold_slot_share@10", "cold_exposure_ratio@10", "novelty@10", "coverage@10",
        "cold_recall@10_lift", "warm_recall@10_lift"]
SUPPORT = ["n_impressions", "n_users", "n_clicks", "n_cold_click_impressions", "n_cold_clicks",
           "n_warm_click_impressions", "n_zero_click_impressions", "n_within_cold_impressions"]
DELTA_METRICS = ["ndcg@10", "auc", "cold_recall@10", "warm_recall@10", "cold_auc_within"]

## 3. Headline table: every model, train / validation / test

Train is shown because the report must give training and testing performance; for trained models the train numbers are in-sample. Deltas are model minus reference on the same impressions, with user-clustered intervals.

In [ ]:
rows = []
for split in SPLITS:
    for model in MODELS:
        if not has(model, split):
            continue
        res = E.summarise(per_imp(model, split))
        res["coverage@10"] = E.catalogue_coverage(meta[split], scores(model, split))
        rows.append({"split": split, "model": model, "kind": KIND.get(model, "ablation" if model in ABLATION_MODELS else "other"),
                     **{k: res.get(k, np.nan) for k in HEAD + SUPPORT}})
        for k in HEAD:
            v = res.get(k, np.nan)
            X.number(f"{split}.{model}.{k}", v)
            X.number(f"{split}.{model}.{k}.per100", 100 * v, "{:.0f}")
headline = pd.DataFrame(rows).set_index(["split", "model"])
# strongest simple baseline, chosen on validation nDCG@10; exported under the alias "bestbase"
BEST_BASE = headline.loc["val"].query("kind == 'baseline'")["ndcg@10"].idxmax()
X.number("setting.best_baseline", BEST_BASE)
for split in SPLITS:
    if (split, BEST_BASE) in headline.index:
        for k in HEAD:
            v = headline.loc[(split, BEST_BASE), k]
            X.number(f"{split}.bestbase.{k}", v)
            X.number(f"{split}.bestbase.{k}.per100", 100 * v, "{:.0f}")
print("strongest simple baseline (validation nDCG@10):", BEST_BASE)
X.table(headline, "headline_all_models", f"All models, cold threshold {PRIMARY_T}; support columns give eligible impressions and clicks")
show = ["kind", "auc", "ndcg@10", "cold_recall@10", "warm_recall@10", "zero_recall@10", "cold_auc_within",
        "n_impressions", "n_cold_click_impressions"]
display(headline[show].round(4))

In [ ]:
delta_rows = []
for split in EVAL_SPLITS:
    for model in [m for m in MODELS if KIND.get(m) in ("main", "reference", "baseline")]:
        for ref in REFERENCES:
            if model == ref or not (has(model, split) and has(ref, split)):
                continue
            d = E.compare(per_imp(ref, split), per_imp(model, split), DELTA_METRICS, n_boot=N_BOOT, seed=BOOT_SEED)
            for metric, r in d.iterrows():
                delta_rows.append({"split": split, "model": model, "reference": ref, "metric": metric, **r.to_dict()})
                stem = f"{split}.delta.{model}_vs_{ref}.{metric}"
                X.number(stem, r["delta"], "{:+.4f}")
                X.number(stem + ".ci", f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]")
                X.number(stem + ".per100", 100 * r["delta"], "{:+.1f}")
deltas = pd.DataFrame(delta_rows)
X.table(deltas.set_index(["split", "model", "reference", "metric"]), "headline_deltas",
        "Model minus reference, paired, 95% user-clustered bootstrap")
main_vs = deltas[(deltas["model"] == MAIN) & (deltas["split"] == REPORT_SPLIT)]
display(main_vs.set_index(["reference", "metric"])[["base", "other", "delta", "ci_low", "ci_high", "n_impressions"]].round(4))
if len(main_vs):
    fig = P.forest_plot(main_vs.assign(model=main_vs["reference"].map(lambda r: f"vs {r}")), label="model",
                        title=f"{MAIN} minus each reference ({REPORT_SPLIT}), 95% user-clustered interval")
    P.save(fig, X.figure_stem("fig_forest_main_vs_references", "Main model against the three references"))
    plt.show()

## 4. Threshold robustness: cold / warm / zero-exposure at t = 5, 10, 20

Same models, same scores; only the cold definition moves. A conclusion **survives** if the main-minus-reference delta on cold Recall@10 has the same sign at every threshold and its interval excludes zero at each.

In [ ]:
COLS_T = ["cold_recall@10", "cold_hit@10", "warm_recall@10", "zero_recall@10", "cold_auc_within", "cold_ndcg@10",
          "n_cold_click_impressions", "n_cold_clicks", "n_within_cold_impressions"]
rob_rows, rob_delta = [], []
headline_models = [m for m in MODELS if KIND.get(m) in ("main", "reference", "baseline")]
for split in EVAL_SPLITS:
    for t in THRESHOLDS:
        pis = {m: per_imp(m, split, t) for m in headline_models if has(m, split)}
        for m, pi in pis.items():
            res = E.summarise(pi)
            rob_rows.append({"split": split, "threshold": t, "model": m, **{k: res.get(k) for k in COLS_T}})
        if MAIN in pis:
            for ref in REFERENCES:
                if ref in pis:
                    d = E.compare(pis[ref], pis[MAIN], ["cold_recall@10", "warm_recall@10"], n_boot=N_BOOT, seed=BOOT_SEED)
                    for metric, r in d.iterrows():
                        rob_delta.append({"split": split, "threshold": t, "reference": ref, "metric": metric, **r.to_dict()})
robust = pd.DataFrame(rob_rows).set_index(["split", "threshold", "model"])
robust_delta = pd.DataFrame(rob_delta)
X.table(robust, "threshold_robustness", "Cold, warm and zero-exposure metrics at thresholds 5, 10, 20")
display(robust.round(4))
if len(robust_delta):
    sig = (robust_delta["ci_low"] > 0) | (robust_delta["ci_high"] < 0)
    survive = (robust_delta.assign(sign=np.sign(robust_delta["delta"]), sig=sig)
               .groupby(["split", "reference", "metric"])
               .agg(signs=("sign", lambda s: "".join("+" if v > 0 else "-" for v in s)),
                    all_significant=("sig", "all"), same_sign=("sign", lambda s: s.nunique() == 1)))
    survive["survives"] = survive["all_significant"] & survive["same_sign"]
    X.table(robust_delta.set_index(["split", "threshold", "reference", "metric"]), "threshold_robustness_deltas",
            f"{MAIN} minus reference at each threshold")
    X.table(survive, "threshold_survival", "Which conclusions survive at 5, 10 and 20")
    display(survive)

## 5. Metric robustness: does the conclusion hold when popularity is removed?

Plain nDCG mixes cold and warm and rewards popular items. The same comparison is repeated with metrics that do not: within-cold AUC (no warm item to lean on), cold-only nDCG, and popularity-debiased nDCG (click gain weighted by 1 / (prior exposures + 1)^γ, γ = 0.5 and 1).

In [ ]:
ROBUST_METRICS = ["ndcg@10", "auc", "cold_recall@10", "cold_auc_within", "cold_auc_vs_all", "cold_ndcg@10",
                  "popdebiased_ndcg@10_g0.5", "popdebiased_ndcg@10_g1", "cold_slot_share@10"]
mr = []
for split in EVAL_SPLITS:
    for ref in REFERENCES:
        if has(MAIN, split) and has(ref, split):
            d = E.compare(per_imp(ref, split), per_imp(MAIN, split), ROBUST_METRICS, n_boot=N_BOOT, seed=BOOT_SEED)
            mr.append(d.reset_index().assign(split=split, reference=ref))
metric_robust = pd.concat(mr, ignore_index=True) if mr else pd.DataFrame()
if len(metric_robust):
    metric_robust["verdict"] = np.select([metric_robust["ci_low"] > 0, metric_robust["ci_high"] < 0],
                                         ["main better", "main worse"], "no detectable difference")
    X.table(metric_robust.set_index(["split", "reference", "metric"]), "metric_robustness",
            f"{MAIN} minus reference under each metric family")
    display(metric_robust.pivot_table(index="metric", columns=["split", "reference"], values="verdict", aggfunc="first"))

### 5b. Layer 4: who gets exposure (the rich-get-richer loop)

Articles are grouped by how often they were shown in the split: the most-shown 1%, the middle, and the least-shown 80%. *Logged* is the share of all impressions MSN gave each group; each model column is the share of top-10 slots it would give them.

In [ ]:
split = REPORT_SPLIT
conc_models = list(dict.fromkeys(m for m in [MAIN, "nrms_g10", BEST_BASE, "candidate_smoothed_ctr", "popularity", "random"]
                                 if has(m, split)))
conc = D.exposure_concentration(meta[split], {m: scores(m, split) for m in conc_models})
X.table(conc, "exposure_concentration", f"Share of exposure by article popularity group ({split}): logged vs each model's top 10")
for col in conc.columns:
    for idx, name in zip(conc.index, ["top", "middle", "bottom"]):
        X.number(f"conc.{split}.{col}.{name}", conc.loc[idx, col], "{:.1%}")
        if col == BEST_BASE:
            X.number(f"conc.{split}.bestbase.{name}", conc.loc[idx, col], "{:.1%}")
display(conc.round(4))

## 6. Balance: the cold-vs-warm trade-off

Not sacrificing warm articles to help cold ones. Three pieces of evidence:

1. Every model and ablation as a point (warm Recall@10, cold Recall@10).
2. A **slot-quota re-ranker** on saved scores: reserve q = 0, 10, 20, 30% of the top 10 for the best-scoring cold candidates. No retraining.
3. A **balance rule**: maximise cold Recall@10 subject to warm Recall@10 ≥ (1 − x) × the best baseline's warm Recall@10, for x = 1, 2 and 5%. Computed on validation.


In [ ]:
split = "val"
pts = headline.loc[split].reset_index()[["model", "kind", "cold_recall@10", "warm_recall@10", "ndcg@10"]]
curves, sweeps = {}, {}
for m in [MAIN, "nrms_g10", "content_smoothed_ctr"]:
    if has(m, split):
        tab, _ = D.quota_sweep(meta[split], scores(m, split), QUOTAS, PRIMARY_T)
        sweeps[m], curves[m] = tab, tab
sweep_table = pd.concat({m: t[["cold_recall@10", "warm_recall@10", "ndcg@10", "cold_slot_share@10", "zero_recall@10"]]
                         for m, t in sweeps.items()}, names=["model", "quota"])
X.table(sweep_table, "balance_quota_sweep", "Slot-quota re-ranking on validation (q = share of top-10 reserved for cold)")
display(sweep_table.round(4))

fig = P.tradeoff_plot(pts, curves, title="Cold vs warm Recall@10 (validation)")
P.save(fig, X.figure_stem("fig_tradeoff_val", "Cold-vs-warm trade-off with slot-quota curves"))
plt.show()

In [ ]:
baselines = pts[pts["kind"] == "baseline"].set_index("model")
REF_WARM_MODEL = baselines["warm_recall@10"].idxmax()
ref_warm = baselines.loc[REF_WARM_MODEL, "warm_recall@10"]
candidates = pd.concat([pts.set_index("model")[["cold_recall@10", "warm_recall@10"]]] +
                       [t[["cold_recall@10", "warm_recall@10"]].rename(index=lambda q, m=m: f"{m} q={q:.0%}")
                        for m, t in sweeps.items()])
candidates["pareto"] = D.pareto_front(candidates)
rule = pd.DataFrame([{"max_warm_drop": x, "warm_floor": ref_warm * (1 - x),
                      "selected": D.balance_select(candidates, ref_warm, x)} for x in MAX_WARM_DROP])
rule = rule.merge(candidates, left_on="selected", right_index=True, how="left")
print(f"warm reference: best baseline on validation = {REF_WARM_MODEL} (warm Recall@10 {ref_warm:.4f})")
X.table(rule.set_index("max_warm_drop"), "balance_rule", "Balance rule: highest cold Recall@10 with warm Recall@10 within x of the best baseline")
X.table(candidates, "balance_candidates", "All models and quota settings, with Pareto flag (validation)")
for _, r in rule.iterrows():
    tag = f"balance.x{int(round(r['max_warm_drop'] * 100))}"
    X.number(f"{tag}.selected", r["selected"])
    X.number(f"{tag}.cold", r["cold_recall@10"])
    X.number(f"{tag}.warm", r["warm_recall@10"])
X.number("balance.ref_model", REF_WARM_MODEL)
X.number("balance.ref_warm", ref_warm)
if MAIN in sweeps:
    for q, r in sweeps[MAIN].iterrows():
        tag = f"balance.quota{int(round(q * 100))}"
        X.number(f"{tag}.cold", r["cold_recall@10"])
        X.number(f"{tag}.warm", r["warm_recall@10"])
        X.number(f"{tag}.cold.per100", 100 * r["cold_recall@10"], "{:.0f}")
        X.number(f"{tag}.warm.per100", 100 * r["warm_recall@10"], "{:.0f}")
display(rule.round(4))

## 7. Ablation analysis (main deep model)

Each step 5 variant is the main model retrained from scratch with one change; the attention news and user encoders are in every variant.

* **Leave-one-out** (`no_*`): full minus one component (T1, T2, T3, U, candidate-aware attention CA, ID masking IDM). A drop means the component carried information the others could not replace.
* **Add-on** (`full_plus_smoothing`): content smoothed CTR added as a feature.
* Components overlap (content reaches the model through the news encoder as well as T2, T3 and U), so contributions do not add up.
* **Decision rule**: a component "helps cold articles" only if removing it lowers cold Recall@10 with the validation interval below zero and the test delta has the same sign. Otherwise "no detectable effect".


In [ ]:
tiers = D.tier_matrix([v for v in FG.ABLATIONS])
tiers["prediction files"] = [ablation_model(v) in MODELS for v in tiers.index]
X.table(tiers.replace({True: "✓", False: ""}), "ablation_variants", "Ablation variants and their feature groups")
display(tiers)

ABL_METRICS = ["ndcg@10", "cold_recall@10", "warm_recall@10", "cold_auc_within", "zero_recall@10"]
abl_level, abl_delta = [], []
for split in EVAL_SPLITS:
    pis = {v: per_imp(m, split) for m, v in ABLATION_MODELS.items() if has(m, split)}
    for v, pi in pis.items():
        res = E.summarise(pi)
        abl_level.append({"split": split, "variant": v, **D.describe_variant(v),
                          **{k: res[k] for k in ABL_METRICS}, "cold_click_impressions": res["n_cold_click_impressions"]})
    if "full" in pis:
        d = D.ablation_deltas(pis, "full", ABL_METRICS, n_boot=N_BOOT, seed=BOOT_SEED)
        abl_delta.append(d.assign(split=split))
    if "u_only" in pis:
        d = D.ablation_deltas({k: v for k, v in pis.items() if k.endswith("_only") or k == "u_only"}, "u_only",
                              ABL_METRICS, n_boot=N_BOOT, seed=BOOT_SEED)
        abl_delta.append(d.assign(split=split))
abl_level = pd.DataFrame(abl_level).set_index(["split", "variant"])
abl_delta = pd.concat(abl_delta, ignore_index=True) if abl_delta else pd.DataFrame()
X.table(abl_level, "ablation_levels", "Ablation variants: metric levels and cold-click support")
display(abl_level.round(4))

In [ ]:
if len(abl_delta):
    main_tab = abl_delta[abl_delta["against"] == "full"].pivot_table(
        index=["split", "variant"], columns="metric", values=["delta", "ci_low", "ci_high"])
    X.table(abl_delta.set_index(["split", "against", "variant", "metric"]), "ablation_deltas",
            "Variant minus full (leave-one-out) and minus u_only (add-one-in), user-clustered 95% intervals")
    for _, r in abl_delta.iterrows():
        stem = f"ablation.{r['split']}.{r['variant']}_vs_{r['against']}.{r['metric']}"
        X.number(stem, r["delta"], "{:+.4f}")
        X.number(stem + ".ci", f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]")
    # decision rule on leave-one-out rows
    cr = abl_delta[(abl_delta["metric"] == "cold_recall@10") & (abl_delta["against"] == "full")].set_index(["variant", "split"])
    dec = []
    for v in sorted({i[0] for i in cr.index}):
        info = D.describe_variant(v)
        if info["kind"] != "leave-out" or (v, "val") not in cr.index:
            continue
        val = cr.loc[(v, "val")]
        test_delta = cr.loc[(v, "test"), "delta"] if (v, "test") in cr.index else None
        dec.append({"variant": v, "tier removed": info["groups"], "val delta": val["delta"],
                    "val CI": f"[{val['ci_low']:+.4f}, {val['ci_high']:+.4f}]",
                    "test delta": test_delta, "decision": D.decision(val["ci_low"], val["ci_high"], test_delta)})
        X.number(f"ablation.{v}.decision", dec[-1]["decision"])
        X.number(f"ablation.{v}.val_delta", val["delta"], "{:+.4f}")
    decisions = pd.DataFrame(dec).set_index("variant") if dec else pd.DataFrame()
    X.table(decisions, "ablation_decisions", "Pre-registered decision rule applied to cold Recall@10")
    display(decisions)
    d = abl_delta[abl_delta["split"] == REPORT_SPLIT].assign(model=lambda t: t["variant"] + " − " + t["against"])
    fig = P.forest_plot(d[d["metric"].isin(["ndcg@10", "cold_recall@10", "warm_recall@10"])],
                        title=f"Ablations ({REPORT_SPLIT}): change against full / u_only")
    P.save(fig, X.figure_stem("fig_ablation_forest", "Ablation deltas with user-clustered intervals"))
    plt.show()

In [ ]:
# Where each component matters: change in Recall@10 by prior-exposure bucket when a tier is removed.
split = REPORT_SPLIT
heat, lo, hi, heat_by_variant = {}, {}, {}, {}
if has(MAIN, split):
    _, long_full = SG.evaluate_by(meta[split], scores(MAIN, split), by="exposure", return_long=True)
    for m, v in ABLATION_MODELS.items():
        info = D.describe_variant(v)
        if info["kind"] != "leave-out" or not has(m, split):
            continue
        _, long_v = SG.evaluate_by(meta[split], scores(m, split), by="exposure", return_long=True)
        sd = SG.segment_deltas(long_full, long_v, "recall@10", n_boot=N_BOOT, seed=BOOT_SEED)
        label = f"without {info['groups']}"
        heat[label], lo[label], hi[label] = sd["delta"], sd["ci_low"], sd["ci_high"]
        heat_by_variant[v] = sd
if heat:
    order = SG.EXPOSURE_LABELS
    H = pd.DataFrame(heat).T.reindex(columns=order)
    L_, U_ = pd.DataFrame(lo).T.reindex(columns=order), pd.DataFrame(hi).T.reindex(columns=order)
    X.table(H, "ablation_heatmap_values", f"Change in Recall@10 by exposure bucket when a tier is removed ({split})")
    fig = P.ablation_heatmap(H, L_, U_)
    P.save(fig, X.figure_stem("fig_ablation_heatmap", "Tier removed × exposure bucket"))
    plt.show()

In [ ]:
# Training noise: spread across seeds, when step 5 writes seeded files (info["seed"]); otherwise state single-seed.
seeded = sorted({m for m in MODELS if "@seed" in m})
if seeded:
    rows = []
    for m in seeded:
        base, seed = m.split("@seed")
        for split in EVAL_SPLITS:
            if has(m, split):
                res = E.summarise(per_imp(m, split))
                rows.append({"variant": base, "seed": seed, "split": split, **{k: res[k] for k in ABL_METRICS}})
    spread = D.seed_spread(pd.DataFrame(rows), ABL_METRICS, by=["split", "variant"])
    X.table(spread, "ablation_seed_spread", "Spread across training seeds")
    display(spread.round(4))
else:
    print("No seeded prediction files: every model is single-seed. State this as a limitation.")

## 8. Segment diagnostics

Candidate-level segments (exposure bucket, category): Recall@10 of clicked candidates in the segment, in full slates. Impression-level segments (user history, slate length, hour): every metric within the segment. Lift is over a random ranker on the same impressions, which makes segments with different slate lengths comparable.

In [ ]:
split = REPORT_SPLIT
SEG_MODELS = list(dict.fromkeys(m for m in (MAIN, "nrms_g10", BEST_BASE, "content_smoothed_ctr", "category_ctr", "popularity")
                                if has(m, split)))
seg_tables = {}
for by in ("exposure", "category"):
    t = pd.concat({m: SG.evaluate_by(meta[split], scores(m, split), by=by) for m in SEG_MODELS}, names=["model"])
    seg_tables[by] = t
    X.table(t, f"segment_{by}", f"Recall@10 of clicked candidates by {by} ({split})")
for by in ("history", "slate", "hour"):
    t = pd.concat({m: SG.evaluate_by(meta[split], scores(m, split), by=by, per_imp=per_imp(m, split),
                                     metrics=["ndcg@10", "auc", "cold_recall@10", "warm_recall@10", "cold_recall@10_lift"])
                   for m in SEG_MODELS}, names=["model"])
    seg_tables[by] = t
    X.table(t, f"segment_{by}", f"Metrics by {by} bucket ({split})")

for by, t in seg_tables.items():
    col = "recall@10" if by in ("exposure", "category") else "cold_recall@10"
    for (m, b), v in t[col].items():
        X.number(f"{split}.seg.{by}.{str(b).replace(' ', '_')}.{'bestbase' if m == BEST_BASE else m}.{col}", v)
        if m == BEST_BASE:
            X.number(f"{split}.seg.{by}.{str(b).replace(' ', '_')}.{m}.{col}", v)
exp = seg_tables["exposure"]["recall@10"].unstack("model").reindex(SG.EXPOSURE_LABELS)
supp = seg_tables["exposure"].xs(SEG_MODELS[0], level="model")["eligible_impressions"]
fig = P.metric_by_bucket(exp[SEG_MODELS[:3]], support=supp, title=f"Recall@10 by prior-exposure bucket ({split})")
P.save(fig, X.figure_stem("fig_recall_by_exposure", "Recall@10 by exposure bucket, top models"))
plt.show()
display(seg_tables["exposure"].round(4))

## 9. Stress slates (secondary result, resampled)

Every impression is rebuilt as a 10-candidate slate with a fixed mix (40% warm, 60% cold), keeping every click and sampling unclicked candidates. This tests whether the ranking holds when cold articles are not a small minority. Three sampling seeds; impressions that cannot meet the mix are dropped and counted. Valid because all models score candidates pointwise. **Label it as resampled in the report.**

In [ ]:
split = REPORT_SPLIT
cold = E.cold_mask(meta[split], PRIMARY_T)
stress_rows, stress_info = [], []
for seed in STRESS["seeds"]:
    mask, info = D.stress_slate_mask(meta[split], cold, STRESS["slate_size"], STRESS["warm_share"], seed)
    stress_info.append(info)
    sub = meta[split][mask].reset_index(drop=True)
    for m in [x for x in MODELS if KIND.get(x) in ("main", "reference", "baseline") and has(x, split)]:
        res = E.evaluate(sub, scores(m, split)[mask], cold_threshold=PRIMARY_T)
        stress_rows.append({"seed": seed, "model": m, **{k: res[k] for k in ["ndcg@10", "auc", "cold_recall@5",
                            "warm_recall@5", "cold_auc_within", "cold_auc_vs_all", "n_impressions"]}})
stress = pd.DataFrame(stress_rows).groupby("model").agg(["mean", "std"])
X.table(stress, "stress_slates", f"Stress slates ({split}, resampled): 10 candidates, 40% warm; mean and SD over 3 seeds")
X.table(pd.DataFrame(stress_info), "stress_slates_info", "Stress-slate construction: impressions kept")
display(pd.DataFrame(stress_info), stress.round(4))

## 10. Error analysis (validation, main model)

Three groups of cold candidates: clicked and ranked in the top 3 (**success**), clicked and ranked in the bottom quarter (**failure**), and unclicked but ranked in the top 3 (**false alarm**). Patterns are computed on all cases; the 30 sampled rows per group are for reading titles. Run on validation so test stays unread for anything but confirmation.

In [ ]:
split = "val"
ATTR_COLS = ["row_id", "entity_count", "entity_embedding_coverage", "article_age_hours", "prior_ctr", "recent_exposures_6h",
             "category_recent_ctr_6h", "subcategory_recent_ctr_6h", "entity_recent_ctr_mean_6h", "semantic_neighbor_ctr",
             "text_similarity", "entity_similarity", "category_affinity", "subcategory_affinity"]
import pyarrow.parquet as pq
feat_cols = pq.read_schema(S.features_path(FEATURE_ROOT, split)).names
attrs = pd.read_parquet(S.features_path(FEATURE_ROOT, split), columns=[c for c in ATTR_COLS if c in feat_cols])
news = None
try:     # titles and abstracts from the raw MIND files, when available
    if DATA_KIND == "REAL":
        import kagglehub
        RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
    else:
        RAW_DIR = os.path.join(FEATURE_ROOT, "raw")
    news = pd.read_csv(os.path.join(RAW_DIR, "news.tsv"), sep="\t", header=None, quoting=3,
                       names=["news_id", "cat", "subcat", "title", "abstract", "url", "te", "ae"])[["news_id", "title", "abstract"]]
    news["abstract_present"] = news["abstract"].fillna("").str.len().gt(0).astype(float)
    news["title_length"] = news["title"].fillna("").str.split().str.len()
except Exception as exc:
    print("raw news.tsv not available, titles skipped:", exc)

cases = D.error_cases(meta[split], scores(MAIN, split), E.cold_mask(meta[split], PRIMARY_T), attrs)
if news is not None:
    cases = cases.merge(news, on="news_id", how="left")
patterns = D.compare_cases(cases)
X.table(patterns["numeric"], "error_patterns_numeric", f"Error analysis ({MAIN}, {split}): attributes by case, all cases")
if "category" in patterns:
    X.table(patterns["category"], "error_patterns_category", "Category mix by case")
samples = D.sample_cases(cases, ERROR_N)
pd.concat(samples, names=["case"]).to_csv(os.path.join(X.tables_dir, "error_samples.csv"))
for c, n in cases["case"].value_counts().items():
    X.number(f"error.{c}.count", int(n), "{:,}")
tierpat = D.tier_patterns(cases)
X.table(tierpat, "error_patterns_by_tier", "Which evidence differs between failures and successes, grouped by tier "
        "(std_diff = standardised mean difference, failure minus success; |0.2| or more is flagged)")
notable = tierpat[tierpat["notable"]]
for i in range(1, 4):
    X.number(f"error.pattern{i}", f"{notable.iloc[i - 1]['reading']} [{notable.iloc[i - 1]['tier']}]"
             if len(notable) >= i else "no further notable difference")
for tier in D.TIER_ATTRS:
    X.number(f"error.tier.{tier.split()[0]}.notable", int(notable["tier"].eq(tier).sum()))
display(tierpat.round(3))
display(patterns["numeric"].round(3))
display(patterns.get("category", pd.DataFrame()).round(3))
for name, s in samples.items():
    print(f"--- {name}: {D.CASES[name]}")
    display(s[[c for c in ("rank", "slate_size", "category", "prior_exposures", "title") if c in s.columns]].head(10))

## 11. Business numbers

Only claims backed by a number above. Clicked cold articles placed in the top 10, per 1,000 impressions, against the strongest reference, and the warm cost. These are offline counts on logged slates: they say how the ranking would have ordered articles users did click, not how many extra clicks a live system would get.

In [ ]:
split = REPORT_SPLIT
biz = []
for ref in dict.fromkeys(REFERENCES + [BEST_BASE]):
    if has(MAIN, split) and has(ref, split):
        b = D.business_impact(per_imp(ref, split), per_imp(MAIN, split), n_boot=N_BOOT, seed=BOOT_SEED)
        biz.append(b.reset_index().assign(model=MAIN, reference=ref))
        for metric, r in b.iterrows():
            for name in [ref] + (["bestbase"] if ref == BEST_BASE else []):
                X.number(f"business.{split}.{MAIN}_vs_{name}.{metric}", r["delta_per_1000"], "{:+.1f}")
                X.number(f"business.{split}.{MAIN}_vs_{name}.{metric}.ci",
                         f"[{r['ci_low']:+.1f}, {r['ci_high']:+.1f}]")
                X.number(f"business.{split}.{name}.{metric}.base", r["base_per_1000"], "{:.1f}")
            X.number(f"business.{split}.{MAIN}.{metric}.level", r["model_per_1000"], "{:.1f}")
if has(MAIN, split) and MAIN in sweeps:
    for q in QUOTAS[1:]:
        _, pis = D.quota_sweep(meta[split], scores(MAIN, split), (q,), PRIMARY_T)
        b = D.business_impact(per_imp(MAIN, split), pis[q], n_boot=N_BOOT, seed=BOOT_SEED)
        biz.append(b.reset_index().assign(model=f"{MAIN} q={q:.0%}", reference=MAIN))
        for metric, r in b.iterrows():
            X.number(f"business.{split}.quota{int(round(q * 100))}.{metric}", r["delta_per_1000"], "{:+.1f}")
business = pd.concat(biz, ignore_index=True) if biz else pd.DataFrame()
if len(business):
    X.table(business.set_index(["model", "reference", "metric"]), "business_clicks_per_1000",
            f"Clicked articles in the top 10 per 1,000 impressions ({split}); offline counts")
    display(business.round(2))

# which categories gain most (cold Recall@10 by category, main vs best baseline by validation nDCG@10)
best_base = BEST_BASE
_, la = SG.evaluate_by(meta[split].assign(cat_cold=np.where(E.cold_mask(meta[split], PRIMARY_T), meta[split]["category"], None)),
                       scores(best_base, split), by="cat_cold", return_long=True)
_, lb = SG.evaluate_by(meta[split].assign(cat_cold=np.where(E.cold_mask(meta[split], PRIMARY_T), meta[split]["category"], None)),
                       scores(MAIN, split), by="cat_cold", return_long=True)
cat_gain = SG.segment_deltas(la, lb, "recall@10", n_boot=N_BOOT, seed=BOOT_SEED).dropna(subset=["delta"]).sort_values("delta", ascending=False)   # categories with no clicked cold article are dropped
X.table(cat_gain, "business_category_gain", f"Cold Recall@10 gain by category, {MAIN} minus {best_base} ({split})")
if len(cat_gain):
    X.number(f"business.{split}.top_category", cat_gain.index[0])
    X.number(f"business.{split}.top_category.gain", cat_gain["delta"].iloc[0], "{:+.3f}")
    X.number(f"business.{split}.top_category.gain.per100", 100 * cat_gain["delta"].iloc[0], "{:+.0f}")
    X.number(f"business.{split}.top_category.ci", f"[{cat_gain['ci_low'].iloc[0]:+.3f}, {cat_gain['ci_high'].iloc[0]:+.3f}]")
    X.number(f"business.{split}.weakest_category", cat_gain.index[-1])
    X.number(f"business.{split}.weakest_category.gain", cat_gain["delta"].iloc[-1], "{:+.3f}")
    X.number(f"business.{split}.n_categories_gain", int((cat_gain["ci_low"] > 0).sum()))
    X.number(f"business.{split}.n_categories_loss", int((cat_gain["ci_high"] < 0).sum()))
    X.number(f"business.{split}.n_categories", len(cat_gain))
display(cat_gain.round(4))

## 12. Answering the research question

> *Does combining T1, T2 and T3 improve personalised ranking for under-exposed articles, and how does each tier's contribution differ between cold and warm articles?*

**Part 1, the combination.** The main model (content and history encoders + T1, T2, T3, U) against the references: the best simple baseline (behaviour only) and Group 10's NRMS (content only). A verdict needs the interval to exclude zero.

**Part 2, each component.** Contribution = main model minus the model retrained without it, so **positive means it helps**. Read the cold and warm columns side by side. "Helps most at" is the exposure bucket where removing it costs the most Recall@10.


In [ ]:
split = REPORT_SPLIT
RQ_METRICS = ["cold_recall@10", "warm_recall@10", "zero_recall@10", "cold_auc_within", "ndcg@10"]
others = [r for r in dict.fromkeys(REFERENCES + [BEST_BASE]) if has(r, split)]
part1 = []
for other in (others if has(MAIN, split) else []):
    d = E.compare(per_imp(other, split), per_imp(MAIN, split), RQ_METRICS, n_boot=N_BOOT, seed=BOOT_SEED)
    verdict = D.tradeoff_verdict(d.loc["cold_recall@10", "ci_low"], d.loc["cold_recall@10", "ci_high"],
                                 d.loc["warm_recall@10", "ci_low"], d.loc["warm_recall@10", "ci_high"])
    row = {"combined model vs": other, "verdict": verdict}
    aliases = [other] + (["bestbase"] if other == BEST_BASE else [])
    for m, r in d.iterrows():
        row[f"Δ {m}"] = r["delta"]
        row[f"{m} CI"] = f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"
        for a in aliases:
            X.number(f"rq.vs_{a}.{m}", r["delta"], "{:+.4f}")
            X.number(f"rq.vs_{a}.{m}.ci", row[f"{m} CI"])
            X.number(f"rq.vs_{a}.{m}.per100", 100 * r["delta"], "{:+.1f}")
    for a in aliases:
        X.number(f"rq.vs_{a}.verdict", verdict)
    part1.append(row)
part1 = pd.DataFrame(part1).set_index("combined model vs") if part1 else pd.DataFrame()
X.table(part1, "rq_part1_combination", f"RQ part 1 ({split}): full model minus each comparison, 95% user-clustered intervals")
display(part1)

contrib = D.tier_contributions(abl_delta, split, metrics=RQ_METRICS) if len(abl_delta) else pd.DataFrame()
for v in contrib.index:
    g = contrib.loc[v, "group removed"]
    if v in heat_by_variant:
        gain = (-heat_by_variant[v]["delta"]).dropna()
        if len(gain):
            contrib.loc[v, "helps most at exposures"] = gain.idxmax()
            contrib.loc[v, "contribution there"] = gain.max()
            X.number(f"rq.tier.{g}.peak_bucket", gain.idxmax())
            X.number(f"rq.tier.{g}.peak_gain", gain.max(), "{:+.3f}")
    for m in RQ_METRICS:
        if f"{m} contribution" in contrib.columns and pd.notna(contrib.loc[v, f"{m} contribution"]):
            c = contrib.loc[v]
            X.number(f"rq.tier.{g}.{m}", c[f"{m} contribution"], "{:+.4f}")
            X.number(f"rq.tier.{g}.{m}.ci", f"[{c[f'{m} ci_low']:+.4f}, {c[f'{m} ci_high']:+.4f}]")
            X.number(f"rq.tier.{g}.{m}.per100", 100 * c[f"{m} contribution"], "{:+.1f}")
            X.number(f"rq.tier.{g}.{m}.reading", c[f"{m} reading"])
if len(contrib):
    X.table(contrib, "rq_part2_tier_contributions",
            f"RQ part 2 ({split}): what each group adds = full minus the model without it (positive = helps)")
    show = ["group removed"] + [f"{m} {x}" for m in ("cold_recall@10", "warm_recall@10", "zero_recall@10")
                                for x in ("contribution", "reading")] + \
           [c for c in ("helps most at exposures", "contribution there") if c in contrib.columns]
    display(contrib[[c for c in show if c in contrib.columns]].round(4))

# one-line plain-English summary per question
summary = []
if len(part1) and BEST_BASE in part1.index:
    summary.append({"question": f"Does the combined model rank clicked cold articles higher than the best simple baseline ({BEST_BASE})?",
                    "answer": part1.loc[BEST_BASE, "verdict"]})
for v in contrib.index:
    c = contrib.loc[v]
    summary.append({"question": f"What does {c['group removed']} add?",
                    "answer": f"cold: {c.get('cold_recall@10 reading', 'n/a')}; warm: {c.get('warm_recall@10 reading', 'n/a')}"
                              + (f"; matters most at {c['helps most at exposures']} prior exposures"
                                 if pd.notna(c.get("helps most at exposures")) else "")})
summary = pd.DataFrame(summary)
X.table(summary, "rq_summary_plain", "Research question in one line per part")
display(summary)

## 13. Export

In [ ]:
X.number("setting.read_test", READ_TEST)
X.number("setting.n_boot", N_BOOT, "{:,}")
X.number("setting.primary_threshold", PRIMARY_T)
X.number("setting.report_split", REPORT_SPLIT)
out = X.finish(os.path.join(REPO_DIR, "docs", "report_sections_template.md"))
print(json.dumps({k: v for k, v in out.items() if k != "missing"}, indent=1))
if out.get("missing"):
    print(f"{len(out['missing'])} template numbers not produced in this run (a prediction file is missing):")
    print(", ".join(out["missing"][:40]))
print(f"tables:  {X.tables_dir}\nfigures: {X.figures_dir}")